In [1]:
import json
import random
from datasets import Dataset, load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch
import numpy as np


In [2]:
# 1. 加载预处理后的数据集
def load_jsonl_data(file_path):
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return data

FILE_PATH = 'dataset/medical_instruction_data.jsonl'

# 加载预处理后的医疗问答数据
medical_data = load_jsonl_data(FILE_PATH)
print(f"已加载 {len(medical_data)} 条样本")

已加载 119397 条样本


In [3]:
# 2. 划分训练集和测试集 (90%训练, 10%测试)
random.seed(42)
random.shuffle(medical_data)

split_index = int(0.9 * len(medical_data))
train_data = medical_data[:split_index]
test_data = medical_data[split_index:]

print(f"训练集大小: {len(train_data)}, 测试集大小: {len(test_data)}")

# 转换为Hugging Face Dataset格式
train_dataset = Dataset.from_list(train_data)
test_dataset = Dataset.from_list(test_data)
train_dataset

训练集大小: 107457, 测试集大小: 11940


Dataset({
    features: ['system', 'conversations'],
    num_rows: 107457
})

In [4]:
# 3. 加载模型和tokenizer
model_path = 'Qwen-7B-Chat'  # 可根据需要替换为其他模型
device_map = "auto"  # 自动分配设备

# 加载tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    trust_remote_code=True,
    use_fast=True
)

if tokenizer.pad_token is None:
    # 尝试使用现有的特殊token
    if tokenizer.eos_token is not None:
        tokenizer.pad_token = tokenizer.eos_token
        print(f"使用eos_token作为pad_token: {tokenizer.pad_token}")
    elif tokenizer.unk_token is not None:
        tokenizer.pad_token = tokenizer.unk_token
        print(f"使用unk_token作为pad_token: {tokenizer.pad_token}")
    elif tokenizer.bos_token is not None:
        tokenizer.pad_token = tokenizer.bos_token
        print(f"使用bos_token作为pad_token: {tokenizer.pad_token}")
    else:
        # 使用现有词汇表中的token作为pad token
        pad_token_id = tokenizer.vocab_size - 1  # 使用最后一个token
        tokenizer.pad_token = tokenizer.decode([pad_token_id])
        print(f"使用词汇表末尾token作为pad_token: {tokenizer.pad_token}")


# 手动设置一个基础模板
if tokenizer.chat_template is None:
    tokenizer.chat_template = (
    "{% for message in messages %}"
        "{% if message['role'] == 'system' %}"
            "<|system|>\n{{ message['content'] }}</s>\n"
        "{% elif message['role'] == 'user' %}"
            "<|user|>\n{{ message['content'] }}</s>\n"
        "{% elif message['role'] == 'assistant' %}"
            "<|assistant|>\n{{ message['content'] }}</s>\n"
        "{% endif %}"
    "{% endfor %}"
    "{% if add_generation_prompt %}"
        "<|assistant|>\n"
    "{% endif %}"
    )

tokenizer.padding_side = "right"  # 填充在右侧

# 直接加载到CPU
print("正在加载模型到CPU...")
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    device_map=None,  # 明确不使用设备映射
    low_cpu_mem_usage=True  # 减少CPU内存占用
)
model = model.to('cpu')  # 确保模型在CPU上


使用词汇表末尾token作为pad_token: <|extra_204|>
正在加载模型到CPU...


The model is automatically converting to bf16 for faster inference. If you want to disable the automatic precision, please manually add bf16/fp16/fp32=True to "AutoModelForCausalLM.from_pretrained".
Try importing flash-attention for faster inference...


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

In [5]:
# 4. 格式化对话函数
def format_conversation(example):
    # 创建消息列表
    messages = [
        {"role": "system", "content": example['system']},
        *example['conversations']
    ]
    
    # 应用聊天模板
    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )
    
    return {"text": formatted_text}

# 应用格式化函数
train_dataset = train_dataset.map(format_conversation, remove_columns=train_dataset.column_names)
test_dataset = test_dataset.map(format_conversation, remove_columns=test_dataset.column_names)


Map:   0%|          | 0/107457 [00:00<?, ? examples/s]

Map:   0%|          | 0/11940 [00:00<?, ? examples/s]

In [6]:
# 5. 数据预处理函数
def preprocess_function(example):
    # 对文本进行分词
    tokenized = tokenizer(
        example["text"],
        truncation=True,
        max_length=1024,  # 根据模型最大上下文长度调整
        padding=False,     # 后续由data_collator统一填充
        return_tensors=None
    )
    
    # 创建标签副本，后续会修改需要忽略的部分
    labels = tokenized["input_ids"].copy()
    
    # 找到assistant开始的位置（计算损失时忽略前面的内容）
    assistant_token_id = tokenizer.convert_tokens_to_ids("<|assistant|>")
    if assistant_token_id is None:
        # 回退策略：使用文本搜索
        content = example["text"]
        assistant_pos = content.find("<|assistant|>")
        if assistant_pos != -1:
            # 使用字符位置近似token位置
            tokenized_full = tokenizer(content, return_tensors=None)
            approx_index = len(tokenizer(content[:assistant_pos], return_tensors=None)["input_ids"])
            last_assistant_index = min(approx_index, len(labels)-1)
        else:
            # 如果没有找到，则使用整个文本
            last_assistant_index = 0
    else:
        assistant_indexes = np.where(np.array(labels) == assistant_token_id)[0]
        last_assistant_index = assistant_indexes[-1] if len(assistant_indexes) > 0 else 0

    # 将assistant之前的内容设为-100（损失计算时忽略）
    labels[:last_assistant_index] = [-100] * last_assistant_index

    return {
        "input_ids": tokenized["input_ids"],
        "attention_mask": tokenized["attention_mask"],
        "labels": labels
    }

# 应用预处理
train_dataset = train_dataset.map(
    preprocess_function,
    remove_columns=["text"],
    desc="Tokenizing train data"
)
test_dataset = test_dataset.map(
    preprocess_function,
    remove_columns=["text"],
    desc="Tokenizing test data"
)

Tokenizing train data:   0%|          | 0/107457 [00:00<?, ? examples/s]

Tokenizing test data:   0%|          | 0/11940 [00:00<?, ? examples/s]

In [7]:
# #6. 配置4-bit量化
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.bfloat16,
#     bnb_4bit_use_double_quant=True
# )

# # 重新加载模型应用量化
# model = AutoModelForCausalLM.from_pretrained(
#     model_path,
#     # quantization_config=bnb_config,
#     device_map=device_map,
#     trust_remote_code=True,
#     torch_dtype=torch.bfloat16
# )

In [1]:
# 7. 准备模型进行k-bit训练
model = prepare_model_for_kbit_training(model)

'''
prepare_model_for_kbit_training(model) 是 Hugging Face peft 库（Parameter-Efficient Fine-Tuning，
参数高效微调）中用于为低比特（k-bit）训练准备模型的关键函数。
它的核心作用是对模型进行必要的修改，使其能够在低精度（如 4-bit、8-bit）状态下稳定进行微调训练，同时尽可能保留模型性能。
'''

NameError: name 'prepare_model_for_kbit_training' is not defined

In [9]:
# 8. 配置LoRA
peft_config = LoraConfig(
    r=4,
    lora_alpha=32,
    target_modules=["c_attn", "c_proj", "w1", "w2"],  # Qwen的注意力模块
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# 应用LoRA
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# 确保模型在正确设备上
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# 确保模型在CPU上
model.to('cpu')
print("模型已加载到CPU")

trainable params: 8,945,664 || all params: 7,730,270,208 || trainable%: 0.1157
模型已加载到CPU


In [10]:
# 9. 设置训练参数
training_args = TrainingArguments(
    output_dir="./qwen-medical-finetune-cpu",
    num_train_epochs=3,
    per_device_train_batch_size=1,  # 减小批处理大小
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,  # 增加梯度累积
    learning_rate=1e-5,  # 降低学习率
    weight_decay=0.01,
    warmup_ratio=0.03,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=400,
    save_total_limit=1,  # 减少保存的检查点
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,
    fp16=False,  # 在CPU上必须关闭FP16
    optim="adamw_torch",  # 使用标准优化器
    report_to="none",
    remove_unused_columns=False,
    logging_dir="./logs",
    gradient_checkpointing=False,  # 关闭梯度检查点（CPU上效果差）
    group_by_length=True,
    dataloader_num_workers=2,  # 减少工作线程
    no_cuda=True  # 明确禁用CUDA
)

C:\Users\ni\MiniConda3\envs\FineTuning_Med\Lib\site-packages\transformers\training_args.py:1609: FutureWarning: using `no_cuda` is deprecated and will be removed in version 5.0 of 🤗 Transformers. Use `use_cpu` instead
  warnings.warn(


In [11]:
# 10. 创建数据收集器
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    pad_to_multiple_of=8,
    padding=True,
    return_tensors="pt"
)

# 11. 创建Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer
)

C:\Users\ni\AppData\Local\Temp\ipykernel_17536\922986418.py:10: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
# 12. 开始训练
print("开始微调训练...")
trainer.train()

开始微调训练...


In [ ]:
# 13. 保存最终模型
model.save_pretrained(training_args.output_dir)
print(f"训练完成！模型已保存到 {training_args.output_dir}")